In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("domain_name", "")
dbutils.widgets.text("sub_domain_name", "")

table_name = dbutils.widgets.get("table_name").strip()
domain_name = dbutils.widgets.get("domain_name").strip()
sub_domain_name = dbutils.widgets.get("sub_domain_name").strip()

# Used only for case-insensitive metadata lookup
table_name_lower = table_name.lower()
domain_name_lower = domain_name.lower()
sub_domain_name_lower = sub_domain_name.lower()

In [0]:
# Future alert implementation
#
# def trigger_alert(event_type, execution_id=None, asset_id=None, message=None):
#     """
#     Future implementation could:
#     - send email
#     - create UI notification
#     - trigger webhook
#     - notify the requesting user
#     """
#     pass

In [0]:
from pyspark.sql import functions as F

CATALOG = "mastercard_card_241614"
SCHEMA = "default"

assets_df = spark.table(
    f"{CATALOG}.{SCHEMA}.assets_m"
)

domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.domains_m"
)

sub_domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.sub_domains_m"
)

In [0]:
asset_df = (
    assets_df
    .filter(
        F.lower(
            F.regexp_extract(
                F.col("s3_path"),
                r"([^/]+)$",
                1
            )
        ) == table_name_lower
    )
)

In [0]:
display(
    asset_df.select(
        "asset_id",
        "asset_name",
        "domain_id",
        "sub_domain_id",
        "s3_path"
    )
)

asset_id,asset_name,domain_id,sub_domain_id,s3_path
28,01_TXN_transaction_lifecycle,1,1,source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle


In [0]:
asset_count = asset_df.count()

if asset_count == 0:
    message = (
        f"No asset found for table='{table_name}', "
        f"domain='{domain_name}', "
        f"sub_domain='{sub_domain_name}'"
    )

    # trigger_alert(
    #     event_type="ASSET_NOT_FOUND",
    #     message=message
    # )

    raise Exception(message)

if asset_count > 1:
    message = (
        f"Multiple assets found for table='{table_name}'. "
        f"Expected exactly one asset."
    )

    # trigger_alert(
    #     event_type="MULTIPLE_ASSETS_FOUND",
    #     message=message
    # )

    raise Exception(message)

asset = asset_df.first()

asset_id = asset.asset_id
asset_domain_id = asset.domain_id
asset_sub_domain_id = asset.sub_domain_id

In [0]:
domain = (
    domains_df
    .filter(
        F.col("domain_id") == asset_domain_id
    )
    .first()
)

if domain is None:
    message = (
        f"Domain metadata missing for asset_id={asset_id}"
    )

    # trigger_alert(
    #     event_type="DOMAIN_METADATA_MISSING",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

if domain.domain_name.lower() != domain_name_lower:
    message = (
        f"Domain mismatch for asset_id={asset_id}. "
        f"Expected='{domain.domain_name}', "
        f"Received='{domain_name}'"
    )

    # trigger_alert(
    #     event_type="DOMAIN_MISMATCH",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
sub_domain = (
    sub_domains_df
    .filter(
        (F.col("sub_domain_id") == asset_sub_domain_id) &
        (F.col("domain_id") == asset_domain_id)
    )
    .first()
)

if sub_domain is None:
    message = (
        f"Sub-domain metadata or domain relationship "
        f"missing for asset_id={asset_id}"
    )

    # trigger_alert(
    #     event_type="SUB_DOMAIN_METADATA_MISSING",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
if sub_domain.sub_domain_name.lower() != sub_domain_name_lower:
    message = (
        f"Sub-domain mismatch for asset_id={asset_id}. "
        f"Expected='{sub_domain.sub_domain_name}', "
        f"Received='{sub_domain_name}'"
    )

    # trigger_alert(
    #     event_type="SUB_DOMAIN_MISMATCH",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
print("✅ Asset validation passed")
print(f"   Asset ID     : {asset_id}")
print(f"   Domain       : {domain.domain_name}")
print(f"   Sub-domain   : {sub_domain.sub_domain_name}")
print(f"   Table        : {table_name}")

✅ Asset validation passed
   Asset ID     : 28
   Domain       : TXN_PROCESSING
   Sub-domain   : TRANSACTION_LIFECYCLE_STATUS
   Table        : 01_TXN_transaction_lifecycle


In [0]:
import pandas as pd

EXCEL_PATH = (
    "/Volumes/mastercard_card_241614/default/"
    "test_data/PCN_DQ_TestDataset (1).xlsx"
)

pdf = pd.read_excel(
    EXCEL_PATH,
    sheet_name=table_name
)

In [0]:
for col in pdf.select_dtypes(include=['object']).columns:
    pdf[col] = pdf[col].where(pdf[col].isna(), pdf[col].astype(str))

df = spark.createDataFrame(pdf)

In [0]:
expected_df = (
    spark.table(
        f"{CATALOG}.{SCHEMA}.asset_columns_m"
    )
    .filter(
        F.col("asset_id") == asset_id
    )
    .orderBy("ordinal_position")
)

In [0]:
expected_columns = [
    row.column_name
    for row in expected_df.select("column_name").collect()
]

actual_columns = df.columns

In [0]:
expected_set = {c.lower() for c in expected_columns}
actual_set = {c.lower() for c in actual_columns}

missing_columns = sorted(
    expected_set - actual_set
)

unexpected_columns = sorted(
    actual_set - expected_set
)

In [0]:
print(expected_columns)
print(actual_columns)

['TRANSACTION_ID', 'LIFECYCLE_STAGE', 'TRANSACTION_TYPE', 'AMOUNT', 'CURRENCY_CODE', 'INITIATED_AT', 'AUTHORIZED_AT', 'CLEARED_AT', 'SETTLED_AT', 'CHANNEL', 'IS_CROSS_BORDER', 'NETWORK', 'PAN_LAST4', 'DQ_FLAG']
['TRANSACTION_ID', 'LIFECYCLE_STAGE', 'TRANSACTION_TYPE', 'AMOUNT', 'CURRENCY_CODE', 'INITIATED_AT', 'AUTHORIZED_AT', 'CLEARED_AT', 'SETTLED_AT', 'CHANNEL', 'IS_CROSS_BORDER', 'NETWORK', 'PAN_LAST4', 'DQ_FLAG']


In [0]:
if missing_columns or unexpected_columns:

    message = (
        f"Schema validation failed for asset_id={asset_id}. "
        f"Missing columns={missing_columns}, "
        f"Unexpected columns={unexpected_columns}"
    )

    # trigger_alert(
    #     event_type="SCHEMA_VALIDATION_FAILED",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
print("✅ Schema validation passed")
print(f"   Asset ID     : {asset_id}")
print(f"   Domain       : {domain_name}")
print(f"   Sub-domain   : {sub_domain_name}")
print(f"   Table        : {table_name}")
print(f"   Columns      : {len(actual_columns)}")

✅ Schema validation passed
   Asset ID     : 28
   Domain       : TXN_PROCESSING
   Sub-domain   : TRANSACTION_LIFECYCLE_STATUS
   Table        : 01_TXN_transaction_lifecycle
   Columns      : 14


In [0]:
dbutils.notebook.exit(
    "SCHEMA_VALIDATION_PASSED"
)